# Notebook 1: Exploring the Data

**Dataset:** Survey of Consumer Finances (SCF) 2019  
**Focus:** Households that were turned down for credit or feared being denied credit (`TURNFEAR == 1`)

In this notebook we:
- Load the SCF 2019 dataset
- Subset to households with `TURNFEAR == 1`
- Explore demographics, income, assets, and debt
- Compute correlations and visualize distributions

## 0. Colab Setup

Run this cell first when working in Google Colab. It installs dependencies and mounts your Google Drive so the data file is accessible.

> **Before running:** Upload the `customer-segmentation/` folder to your Google Drive (or clone your GitHub repo).

In [ ]:
# ── Clone the GitHub repository ────────────────────────────────────────────
# Replace the URL below with your actual GitHub repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'

!git clone {REPO_URL}

# ── Set working directory to the project folder ────────────────────────────
import os

# The notebooks live inside customer-segmentation/ within your repo
# Replace YOUR_REPO_NAME with the actual cloned folder name
os.chdir('/content/YOUR_REPO_NAME/customer-segmentation')

# ── Install required packages ──────────────────────────────────────────────
!pip install pandas numpy matplotlib seaborn plotly scikit-learn scipy --quiet

print(f'Working directory: {os.getcwd()}')
print(f'Data file exists: {os.path.exists("data/SCF_2019.csv.gz")}')

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '${:,.2f}'.format)
pd.set_option('display.max_columns', 50)

print('Libraries loaded successfully!')

## 2. Load the Dataset

> **Note:** Download the SCF 2019 dataset from the Federal Reserve:
> https://www.federalreserve.gov/econres/files/scfp2019s.zip
> Extract and save as `data/SCF_2019.csv.gz`
>
> The SCF uses **multiple imputation** (5 implicates per household). We keep every 5th row (implicate 1) to avoid duplication.

In [ ]:
# Load the compressed dataset
df = pd.read_csv('data/SCF_2019.csv.gz', compression='gzip', index_col=0)

# Keep only implicate 1 (every 5th row starting at index 0)
df = df.iloc[::5].reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Number of households: {len(df):,}')
df.head()

## 3. Subset: Households Turned Down or Fearing Denial of Credit

In [ ]:
# Create mask for households turned down or fearing credit denial
mask = df['TURNFEAR'] == 1

# Create subset dataframe
df_fear = df[mask].copy().reset_index(drop=True)

print(f'Total households:              {len(df):,}')
print(f'TURNFEAR == 1 households:      {len(df_fear):,}')
print(f'Percentage of total:           {len(df_fear)/len(df)*100:.1f}%')

df_fear.head()

## 4. Summary Statistics

In [ ]:
# Key financial columns to explore
key_cols = ['INCOME', 'NETWORTH', 'ASSET', 'DEBT', 'HOUSES', 'MRTHEL',
            'CCBAL', 'SAVBND', 'RETQLIQ', 'NHNFIN', 'AGE', 'EDUC']

# Summary statistics
summary = df_fear[key_cols].describe().T
summary['median'] = df_fear[key_cols].median()
summary = summary[['count', 'mean', 'median', 'std', 'min', 'max']]
summary.columns = ['Count', 'Mean', 'Median', 'Std Dev', 'Min', 'Max']
summary

## 5. Age Distribution

In [ ]:
# Age series
age_series = df_fear['AGE']

fig = px.histogram(
    df_fear,
    x='AGE',
    nbins=30,
    title='Age Distribution of Households Turned Down for Credit',
    labels={'AGE': 'Age of Household Head', 'count': 'Number of Households'},
    color_discrete_sequence=['#636EFA']
)
fig.add_vline(
    x=age_series.median(),
    line_dash='dash',
    line_color='red',
    annotation_text=f'Median: {age_series.median():.0f}'
)
fig.update_layout(bargap=0.05)
fig.show()

print(f'Mean age:   {age_series.mean():.1f} years')
print(f'Median age: {age_series.median():.1f} years')
print(f'Std dev:    {age_series.std():.1f} years')

## 6. Income Distribution

In [ ]:
# Income series — filter extreme outliers for visualization
income_series = df_fear['INCOME']
income_trimmed = income_series[income_series < income_series.quantile(0.99)]

fig = px.histogram(
    x=income_trimmed,
    nbins=50,
    title='Income Distribution (households turned down for credit, excluding top 1%)',
    labels={'x': 'Annual Household Income ($)', 'count': 'Number of Households'},
    color_discrete_sequence=['#EF553B']
)
fig.add_vline(
    x=income_series.median(),
    line_dash='dash',
    line_color='navy',
    annotation_text=f'Median: ${income_series.median():,.0f}'
)
fig.update_layout(bargap=0.05)
fig.show()

print(f'Mean income:   ${income_series.mean():,.0f}')
print(f'Median income: ${income_series.median():,.0f}')
print(f'Std dev:       ${income_series.std():,.0f}')

## 7. Net Worth Distribution

In [ ]:
# Net worth series
networth_series = df_fear['NETWORTH']
networth_trimmed = networth_series[
    (networth_series > networth_series.quantile(0.01)) &
    (networth_series < networth_series.quantile(0.99))
]

fig = px.histogram(
    x=networth_trimmed,
    nbins=50,
    title='Net Worth Distribution (excluding bottom/top 1%)',
    labels={'x': 'Net Worth ($)', 'count': 'Number of Households'},
    color_discrete_sequence=['#00CC96']
)
fig.add_vline(
    x=networth_series.median(),
    line_dash='dash',
    line_color='red',
    annotation_text=f'Median: ${networth_series.median():,.0f}'
)
fig.update_layout(bargap=0.05)
fig.show()

print(f'Mean net worth:   ${networth_series.mean():,.0f}')
print(f'Median net worth: ${networth_series.median():,.0f}')
print(f'Negative net worth (households): {(networth_series < 0).sum():,}')

## 8. Education Level Breakdown

In [ ]:
# Education labels
educ_labels = {
    1: 'No High School',
    2: 'High School Diploma',
    3: 'Some College',
    4: 'College Degree',
    5: 'Graduate Degree'
}

educ_series = df_fear['EDUC'].map(educ_labels)
educ_counts = educ_series.value_counts().reset_index()
educ_counts.columns = ['Education Level', 'Count']

# Order by education level
educ_order = list(educ_labels.values())
educ_counts['Education Level'] = pd.Categorical(
    educ_counts['Education Level'], categories=educ_order, ordered=True
)
educ_counts = educ_counts.sort_values('Education Level')

fig = px.bar(
    educ_counts,
    x='Education Level',
    y='Count',
    title='Education Level of Households Turned Down for Credit',
    color='Count',
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis_title='Education Level', yaxis_title='Number of Households')
fig.show()

print(educ_series.value_counts())

## 9. Assets vs. Debt Scatterplot

In [ ]:
# Create dataframe for scatter plot
scatter_df = df_fear[['ASSET', 'DEBT', 'INCOME', 'AGE']].copy()
scatter_df = scatter_df[
    (scatter_df['ASSET'] < scatter_df['ASSET'].quantile(0.98)) &
    (scatter_df['DEBT'] < scatter_df['DEBT'].quantile(0.98))
]

fig = px.scatter(
    scatter_df,
    x='ASSET',
    y='DEBT',
    color='AGE',
    color_continuous_scale='Viridis',
    title='Total Assets vs. Total Debt (colored by Age)',
    labels={'ASSET': 'Total Assets ($)', 'DEBT': 'Total Debt ($)', 'AGE': 'Age'},
    opacity=0.6
)
fig.show()

## 10. Income vs. Net Worth Scatterplot

In [ ]:
# Income vs net worth
scatter2_df = df_fear[['INCOME', 'NETWORTH', 'EDUC']].copy()
scatter2_df['Education'] = scatter2_df['EDUC'].map(educ_labels)
scatter2_df = scatter2_df[
    (scatter2_df['INCOME'] < scatter2_df['INCOME'].quantile(0.97)) &
    (scatter2_df['NETWORTH'] > scatter2_df['NETWORTH'].quantile(0.02)) &
    (scatter2_df['NETWORTH'] < scatter2_df['NETWORTH'].quantile(0.97))
]

fig = px.scatter(
    scatter2_df,
    x='INCOME',
    y='NETWORTH',
    color='Education',
    title='Income vs. Net Worth (colored by Education Level)',
    labels={'INCOME': 'Annual Income ($)', 'NETWORTH': 'Net Worth ($)'},
    opacity=0.7
)
fig.show()

## 11. Credit Card Debt Distribution

In [ ]:
# Credit card debt — households with CC debt
cc_series = df_fear['CCBAL']
cc_with_debt = cc_series[cc_series > 0]

print(f'Households with CC debt: {len(cc_with_debt):,} ({len(cc_with_debt)/len(df_fear)*100:.1f}%)')
print(f'Mean CC balance:         ${cc_with_debt.mean():,.0f}')
print(f'Median CC balance:       ${cc_with_debt.median():,.0f}')

fig = px.histogram(
    x=cc_with_debt[cc_with_debt < cc_with_debt.quantile(0.99)],
    nbins=40,
    title='Credit Card Balance Distribution (households with CC debt, excl. top 1%)',
    labels={'x': 'Credit Card Balance ($)', 'count': 'Number of Households'},
    color_discrete_sequence=['#AB63FA']
)
fig.update_layout(bargap=0.05)
fig.show()

## 12. Mortgage Debt Histogram

In [ ]:
# Mortgage debt for homeowners
mortgage_series = df_fear['MRTHEL']
has_mortgage = mortgage_series[mortgage_series > 0]

print(f'Households with mortgage: {len(has_mortgage):,} ({len(has_mortgage)/len(df_fear)*100:.1f}%)')
print(f'Mean mortgage balance:    ${has_mortgage.mean():,.0f}')
print(f'Median mortgage balance:  ${has_mortgage.median():,.0f}')

fig = px.histogram(
    x=has_mortgage[has_mortgage < has_mortgage.quantile(0.99)],
    nbins=40,
    title='Mortgage Debt Distribution (households with mortgage, excl. top 1%)',
    labels={'x': 'Mortgage Balance ($)', 'count': 'Number of Households'},
    color_discrete_sequence=['#FFA15A']
)
fig.update_layout(bargap=0.05)
fig.show()

## 13. Correlation Matrix

In [ ]:
# Correlation matrix for key financial variables
corr_cols = ['INCOME', 'NETWORTH', 'ASSET', 'DEBT', 'HOUSES',
             'MRTHEL', 'CCBAL', 'RETQLIQ', 'NHNFIN', 'AGE']

corr_matrix = df_fear[corr_cols].corr()

fig = px.imshow(
    corr_matrix,
    title='Correlation Matrix: Key Financial Variables (TURNFEAR == 1)',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    text_auto='.2f'
)
fig.update_layout(width=700, height=700)
fig.show()

## 14. Correlation with TURNFEAR (Full Dataset)

In [ ]:
# Correlation of financial variables with TURNFEAR in full dataset
turnfear_corr = df[corr_cols + ['TURNFEAR']].corr()['TURNFEAR'].drop('TURNFEAR').sort_values()

fig = px.bar(
    x=turnfear_corr.values,
    y=turnfear_corr.index,
    orientation='h',
    title='Correlation of Financial Variables with TURNFEAR',
    labels={'x': 'Pearson Correlation', 'y': 'Variable'},
    color=turnfear_corr.values,
    color_continuous_scale='RdBu'
)
fig.add_vline(x=0, line_color='black', line_width=1)
fig.show()

print('\nCorrelation coefficients with TURNFEAR:')
print(turnfear_corr.to_string())

## 15. Age vs. Income by Credit Fear Status

In [ ]:
# Compare fear vs non-fear groups
comparison_df = df[['AGE', 'INCOME', 'NETWORTH', 'CCBAL', 'DEBT', 'TURNFEAR']].copy()
comparison_df['Credit Status'] = comparison_df['TURNFEAR'].map(
    {1: 'Turned Down / Feared Denial', 0: 'No Credit Issues'}
)

# Filter extremes for visualization
comparison_df = comparison_df[
    comparison_df['INCOME'] < comparison_df['INCOME'].quantile(0.97)
]

fig = px.box(
    comparison_df,
    x='Credit Status',
    y='INCOME',
    color='Credit Status',
    title='Income Distribution: Credit Fear vs. No Credit Issues',
    labels={'INCOME': 'Annual Household Income ($)'},
    color_discrete_map={
        'Turned Down / Feared Denial': '#EF553B',
        'No Credit Issues': '#636EFA'
    }
)
fig.show()

## 16. Summary: Key Findings

In [ ]:
# Summary comparison table
fear_group = df[df['TURNFEAR'] == 1]
no_fear_group = df[df['TURNFEAR'] == 0]

summary_comparison = pd.DataFrame({
    'Metric': ['Count', 'Median Income', 'Median Net Worth', 'Median CC Debt',
               'Median Total Debt', 'Median Age', 'Pct with Mortgage'],
    'Turned Down / Feared Denial': [
        f'{len(fear_group):,}',
        f'${fear_group["INCOME"].median():,.0f}',
        f'${fear_group["NETWORTH"].median():,.0f}',
        f'${fear_group["CCBAL"].median():,.0f}',
        f'${fear_group["DEBT"].median():,.0f}',
        f'{fear_group["AGE"].median():.0f} years',
        f'{(fear_group["MRTHEL"] > 0).mean()*100:.1f}%'
    ],
    'No Credit Issues': [
        f'{len(no_fear_group):,}',
        f'${no_fear_group["INCOME"].median():,.0f}',
        f'${no_fear_group["NETWORTH"].median():,.0f}',
        f'${no_fear_group["CCBAL"].median():,.0f}',
        f'${no_fear_group["DEBT"].median():,.0f}',
        f'{no_fear_group["AGE"].median():.0f} years',
        f'{(no_fear_group["MRTHEL"] > 0).mean()*100:.1f}%'
    ]
})

print('=== KEY FINDINGS ===')
print(summary_comparison.to_string(index=False))